In [1]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8.9 kB/s  0:03:17m eta 0:00:20


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier



In [7]:
X = np.loadtxt('../data/processed/digits4000_normalized.txt', delimiter='\t')
y=np.loadtxt('../data/raw/digits4000_txt/digits4000_digits_labels.txt', delimiter='\t')
X_pca=np.loadtxt('../data/processed/digits4000_pca.txt', delimiter='\t')
X_train = X[2000:4000]  # 训练集 2000个
X_test = X[0:2000]
y_train = y[2000:4000]  # 训练集标签
y_test = y[0:2000]      # 测试集标签
X_train_pca= X[2000:4000]
X_test_pca = X[0:2000]
print(X.shape)
X_pca.shape

(4000, 784)


(4000, 147)

# Algorithms training and testing
>Choose multiple algorithms to train the model.

In [6]:
# KNN(Baseline)
# 基准模型示例
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
print(f"KNN准确率: {knn.score(X_test, y_test):.4f}")

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
print(f"Logistic回归准确率: {lr.score(X_test, y_test):.4f}")


KNN准确率: 0.9060
Logistic回归准确率: 0.8820


In [7]:
#LinearSVC(线性SVM）
linear_svc = LinearSVC(C=0.01, max_iter=1500, random_state=42)
linear_svc.fit(X_train, y_train)
print(f"LinearSVC准确率: {linear_svc.score(X_test, y_test):.4f}")

LinearSVC准确率: 0.8835


In [8]:
#SVC（RBF） 
#Use PCA dataset
svc_model = svm.SVC(
    kernel='rbf',      # 径向基核函数
    C=10,             # 正则化参数
    gamma='scale',    # 核函数系数
    decision_function_shape='ovo',
    random_state=42
)

svc_model.fit(X_train_pca, y_train)
y_pred = svc_model.predict(X_test_pca)

# 准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"核SVC准确率: {accuracy:.4f} ({accuracy*100:.2f}%)")

核SVC准确率: 0.9410 (94.10%)


In [42]:
#Decision Tree and Random Forest
dt = DecisionTreeClassifier(max_depth=20, min_samples_leaf=10,random_state=42)
dt.fit(X_train, y_train)
print(f"决策树准确率: {dt.score(X_test, y_test):.4f}")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"随机森林准确率: {rf.score(X_test, y_test):.4f}")

决策树准确率: 0.6820
随机森林准确率: 0.9165


In [40]:
#集成学习
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)
print(f"GBDT准确率: {gbdt.score(X_test, y_test):.4f}")

xgb = XGBClassifier(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
print(f"XGBoost准确率: {xgb.score(X_test, y_test):.4f}")

GBDT准确率: 0.8770
XGBoost准确率: 0.9010


# Tune hyperparameters
>choose the top2 algorithms to tune,
>and select the best performance of the two algorithms.

In [8]:
# SVC调参
param_grid = {
    'C': [0.01,0.1,1,5,10,100],      
    'gamma': [0.001,0.005,0.01,0.05,0.5,1,5,10], 
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    svm.SVC(random_state=42),
    param_grid,
    cv=5,  
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

# 输出最佳参数和准确率
print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳CV得分: {grid_search.best_score_:.4f}")

# 测试
best_svc = grid_search.best_estimator_
accuracy = best_svc.score(X_test, y_test)
print(f"测试集准确率: {accuracy:.4f}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits
最佳参数: {'C': 5, 'gamma': 0.05, 'kernel': 'rbf'}
最佳CV得分: 0.9415
测试集准确率: 0.9445


In [9]:
# Random forest tuning
param_grid_rf = {
    'n_estimators': [50, 100,150,200],      # 树的数量
    'max_depth': [10,15,20,25, None],         # 最大深度
    'min_samples_split': [2, 5, 10],     # 分裂所需最小样本数
    'min_samples_leaf': [1, 2, 4]        # 叶子节点最小样本数
}

# 使用较少数据加速搜索
X_sample = X_train[:1000]
y_sample = y_train[:1000]

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,                       scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("start searching...")
grid_rf.fit(X_sample, y_sample)

print(f"\n最佳参数: {grid_rf.best_params_}")
print(f"最佳CV得分: {grid_rf.best_score_:.4f}")

# 使用最佳参数训练完整模型
best_rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42)
best_rf.fit(X_train, y_train)
acc_rf = best_rf.score(X_test, y_test)
print(f"测试集准确率: {acc_rf:.4f}")

start searching...
Fitting 5 folds for each of 180 candidates, totalling 900 fits

最佳参数: {'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
最佳CV得分: 0.9720
测试集准确率: 0.9240


- compare the accuracy, SVC(RFB) get the best performance, so we choose SVC as the best model

In [10]:
import joblib    #保存模型
joblib.dump(best_svc,'../models/svm_mnist_model.pkl')

['../models/svm_mnist_model.pkl']

In [11]:
import json #保存元数据
metadata = {
    'model_type': 'SVM',
    'best_params': grid_search.best_params_,
    'accuracy': float(accuracy),
    'training_samples': len(X_train),
    'features': 784,
    'classes': [0, 1]
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)